# BERT 빈칸 채우기 실습

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/slides/v2/ml1/week14/bert/lab/bert.ipynb)

트랜스포머 인코더가 빈칸의 앞뒤를 모두 보고 원래 단어를 맞힌다.

- **파트 1**: 15% 가리기 (80% [MASK] · 10% 무작위 · 10% 그대로)
- **파트 2**: 실제 BERT 로 빈칸 채우기
- **파트 3**: 문맥 벡터 ("bank") · 가중치 수

`None` 으로 비워 둔 `TODO` 줄을 채우고 셀을 차례로 실행하세요.

In [ ]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

## 파트 1. 가리기

In [ ]:
def mask_tokens(tokens, vocab, rate=0.15, seed=0):
    rng = np.random.default_rng(seed)
    out, targets = list(tokens), {}
    for i, t in enumerate(tokens):
        if rng.random() < rate:                          # choose 15% of the positions
            targets[i] = t
            r = rng.random()
            if r < 0.8:
                out[i] = None  # TODO: 80%: replace with [MASK]
            elif r < 0.9:
                out[i] = rng.choice(vocab)               # 10%: a random word
            # else 10%: keep the original word
    return out, targets

sentence = "the cat sat on the mat and slept because it was a very long and tiring day for the little cat".split()
vocab = ["apple", "run", "blue", "music", "river", "seven"]
masked, targets = mask_tokens(sentence, vocab)
print(" ".join(masked)); print("to predict:", targets)

# On many sentences the split is close to 15% / 80-10-10
picked = masked_n = 0
for s in range(2000):
    m, tg = mask_tokens(sentence, vocab, seed=s)
    picked += len(tg); masked_n += sum(m[i] == "[MASK]" for i in tg)
print(f"picked {picked / (2000 * len(sentence)):.1%} of tokens, {masked_n / picked:.1%} of them became [MASK]")

**예상 결과**

```
the cat [MASK] on the mat and slept because it music a very long and tiring day [MASK] the little cat
to predict: {2: 'sat', 10: 'was', 17: 'for'}
picked 15.2% of tokens, 79.3% of them became [MASK]
```

## 파트 2. 실제 BERT

In [ ]:
tok = AutoTokenizer.from_pretrained("bert-base-uncased")
mlm = AutoModelForMaskedLM.from_pretrained("bert-base-uncased").eval()

def fill_mask(sentence, tok, model, k=3):
    t = tok(sentence, return_tensors="pt")
    pos = (t["input_ids"][0] == tok.mask_token_id).nonzero()[0, 0]
    with torch.no_grad():
        probs = None  # TODO: softmax over the vocabulary at the [MASK] position
    top = probs.topk(k)
    return [(tok.decode([int(i)]).strip(), round(float(p) * 100, 1)) for p, i in zip(top.values, top.indices)]

for s in ["I went to the [MASK] to buy some bread.", "I went to the [MASK] to see a doctor.",
          "The capital of France is [MASK].", "She is a [MASK] at the hospital.", "He is a [MASK] at the hospital."]:
    print(f"{s:42s}", fill_mask(s, tok, mlm))

**예상 결과**

```
I went to the [MASK] to buy some bread.    [('market', 78.0), ('store', 5.6), ('bakery', 4.7)]
I went to the [MASK] to see a doctor.      [('hospital', 81.0), ('clinic', 3.3), ('bathroom', 2.7)]
The capital of France is [MASK].           [('paris', 41.7), ('lille', 7.1), ('lyon', 6.3)]
She is a [MASK] at the hospital.           [('nurse', 60.9), ('patient', 11.0), ('doctor', 10.9)]
He is a [MASK] at the hospital.            [('doctor', 28.7), ('patient', 22.5), ('nurse', 17.5)]
```

## 파트 3. 문맥 벡터 · 가중치 수

In [ ]:
def word_vector(sentence, word, tok, model):
    t = tok(sentence, return_tensors="pt")
    with torch.no_grad():
        h = model.bert(**t).last_hidden_state[0]         # one 768-number vector per token
    k = tok.convert_ids_to_tokens(t["input_ids"][0]).index(word)
    v = h[k].numpy()
    return v / np.linalg.norm(v)

sents = ["I deposited money at the bank.", "The bank approved my loan.",
         "We sat on the bank of the river.", "The river bank was covered with grass."]
V = np.array([word_vector(s, "bank", tok, mlm) for s in sents])
print((V @ V.T).round(2))

**예상 결과**

```
[[1.   0.83 0.54 0.52]
 [0.83 1.   0.42 0.49]
 [0.54 0.42 1.   0.71]
 [0.52 0.49 0.71 1.  ]]
```

In [ ]:
bert = mlm.bert
count = lambda m: sum(p.numel() for p in m.parameters())
print("word table :", f"{bert.embeddings.word_embeddings.weight.numel():,}")
print("one block  :", f"{count(bert.encoder.layer[0]):,}", "  12 d^2 =", f"{12 * 768 ** 2:,}")
print("total      :", f"{count(bert):,}")

**예상 결과**

```
word table : 23,440,896
one block  : 7,087,872   12 d^2 = 7,077,888
total      : 108,891,648
```

**확인 질문**
1. 파트 1 에서 고른 토큰을 전부 [MASK] 로 바꾸지 않는 이유는?
2. 파트 2 에 여러분의 문장을 넣어 보라. 앞만 보면 맞히기 어렵고 뒤를 봐야 맞힐 수 있는 문장을 만들어 보라.
3. 파트 3 에서 은행끼리 · 강둑끼리 유사도가 더 높은 이유는? word2vec 이라면?
4. (도전) "She is a [MASK] at the hospital." 의 편향을 줄이려면 어떤 방법이 있을까?

모델: J. Devlin 외 (2018) BERT, Hugging Face `bert-base-uncased`.